In [ ]:
import pandas as pd

df = pd.read_csv('Coffee Shop Sales.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")


In [ ]:
print("\nDate range:")
print(df['transaction_date'].min(), "to", df['transaction_date'].max())

print("\nProduct categories:")
print(df['product_category'].value_counts())

print("\nStore locations:")
print(df['store_location'].value_counts())

In [23]:
import pandas as pd
import numpy as np
 
# ── 1. LOAD ───────────────────────────────────────────────────────────────────
df = pd.read_csv('Coffee Shop Sales.csv')
trends = pd.read_csv('time_series_MY_20210402-1121_20260402-1121.csv',
                     skiprows=0, header=0)
trends.columns = ['month', 'zus_interest']
trends['month'] = pd.to_datetime(trends['month'])
 
print(f"Raw rows: {len(df)}")

Raw rows: 149116


In [24]:
# ── 2. PARSE DATETIME ─────────────────────────────────────────────────────────
df['transaction_date'] = pd.to_datetime(df['transaction_date'], dayfirst=True)
df['hour']        = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.hour
df['day_of_week'] = df['transaction_date'].dt.day_name()

In [25]:
# ── 3. REMAP DATES → OCT 2025 – MAR 2026 ─────────────────────────────────────
df['transaction_date'] = df['transaction_date'] + pd.DateOffset(months=33)
print(f"Remapped date range: {df['transaction_date'].min().date()} -> {df['transaction_date'].max().date()}")

Remapped date range: 2025-10-01 -> 2026-03-30


In [26]:
# ── 4. REMAP LOCATIONS → KL ZONES ────────────────────────────────────────────
location_map = {
    "Hell's Kitchen" : "KLCC / Bukit Bintang",
    "Astoria"        : "Subang / Shah Alam",
    "Lower Manhattan": "Puchong / Cyberjaya"
}
df['store_location'] = df['store_location'].str.strip()
df['outlet_zone'] = df['store_location'].map(location_map)
 
print(f"\nOutlet zone check (NaN = unmapped):")
print(df['outlet_zone'].value_counts(dropna=False))


Outlet zone check (NaN = unmapped):
outlet_zone
KLCC / Bukit Bintang    50735
Subang / Shah Alam      50599
Puchong / Cyberjaya     47782
Name: count, dtype: int64


In [27]:
# ── 5. REMAP PRODUCT CATEGORIES → ZUS-STYLE ──────────────────────────────────
category_map = {
    'Coffee'              : 'Spanish Latte',
    'Tea'                 : 'Buttermilk Latte',
    'Drinking Chocolate'  : 'Roasted Hazelnut Chocolate',
    'Flavours'            : 'Green Tea Latte',
    'Bakery'              : 'Snack / Pastry',
    'Coffee beans'        : 'Merchandise',
    'Loose Tea'           : 'Matcha Strawberry Latte',
    'Branded'             : 'Merchandise',
    'Packaged Chocolate'  : 'Merchandise'
}
df['zus_category'] = df['product_category'].map(category_map)
 
# Focus on beverages only (what matters for surge/wait time analysis)
df['product_category'] = df['product_category'].str.strip()
df['zus_category'] = df['product_category'].map(category_map)
df = df[df['zus_category'] != 'Merchandise'].copy()
df = df.reset_index(drop=True)
print(f"\nRows after removing merchandise: {len(df)}")
 


Rows after removing merchandise: 146129


In [28]:
# ── 6. ADD ORDER CHANNEL ──────────────────────────────────────────────────────
np.random.seed(42)
 
def assign_channel(hour, dow):
    if hour in range(11, 15) or hour in range(18, 22):
        probs = [0.50, 0.30, 0.20]
    elif dow in ['Saturday', 'Sunday'] and hour in range(13, 18):
        probs = [0.25, 0.20, 0.55]
    elif hour in range(7, 11):
        probs = [0.20, 0.15, 0.65]
    else:
        probs = [0.35, 0.25, 0.40]
    return np.random.choice(['ShopeeFood', 'GrabFood', 'Walk-in'], p=probs)
 
df['order_channel'] = [assign_channel(h, d) for h, d in zip(df['hour'], df['day_of_week'])]

In [29]:
# ── 7. ADD MALAYSIAN HOLIDAY FLAGS ────────────────────────────────────────────
holidays = {
    'Raya 2026'      : pd.date_range('2026-03-28', '2026-04-02'),
    'CNY 2026'       : pd.date_range('2026-01-28', '2026-02-01'),
    'New Year 2026'  : pd.date_range('2025-12-31', '2026-01-02'),
    'Christmas 2025' : pd.date_range('2025-12-24', '2025-12-26'),
    'Deepavali 2025' : pd.date_range('2025-10-20', '2025-10-21'),
}
 
date_to_holiday = {}
for name, daterange in holidays.items():
    for d in daterange:
        date_to_holiday[d] = name
 
def get_occasion(date):
    if date in date_to_holiday:
        return date_to_holiday[date]
    if date.day >= 25 or date.day <= 1:
        return 'Payday Week'
    return 'Normal'
 
df['occasion'] = df['transaction_date'].apply(get_occasion)

In [30]:
# ── 8. ADD DRINK COMPLEXITY ───────────────────────────────────────────────────
# Affects prep time — blended/layered drinks take longer
complexity_map = {
    'Spanish Latte'        : 1,
    'Buttermilk Latte'   : 3,
    'Roasted Hazelnut Chocolate'  : 2,
    'Green Tea Latte'    : 2,
    'Snack / Pastry'    : 1,
    'Matcha Strawberry Latte': 3
}
df['drink_complexity'] = df['zus_category'].map(complexity_map)

In [31]:
# ── 9. HOURLY ORDER VOLUME + PREP TIME ───────────────────────────────────────
df['hourly_order_volume'] = df.groupby(
    ['transaction_date', 'store_id', 'hour'])['transaction_id'].transform('count')
 
def sim_prep(complexity, volume, occasion):
    base         = complexity * 3.5
    surge        = min(volume / 15, 3.0)
    holiday_bump = 1.5 if occasion not in ['Normal', 'Payday Week'] else 1.0
    noise        = np.random.normal(0, 1.5)
    return round(max(2, base * surge * holiday_bump + noise), 1)
 
df['prep_time_mins'] = [
    sim_prep(c, v, o)
    for c, v, o in zip(df['drink_complexity'], df['hourly_order_volume'], df['occasion'])
]

In [32]:
# ── 10. MERGE GOOGLE TRENDS ───────────────────────────────────────────────────
df['trends_month'] = df['transaction_date'].dt.to_period('M').dt.to_timestamp()
df = df.merge(trends, left_on='trends_month', right_on='month', how='left')
df['zus_interest'] = df['zus_interest'].fillna(50)

In [33]:
# ── 11. FINAL COLUMNS ─────────────────────────────────────────────────────────
final_cols = [
    'transaction_id', 'transaction_date', 'hour', 'day_of_week',
    'outlet_zone', 'store_id',
    'zus_category', 'product_type', 'product_detail',
    'transaction_qty', 'unit_price',
    'order_channel', 'occasion',
    'drink_complexity', 'hourly_order_volume', 'prep_time_mins',
    'zus_interest'
]
df_final = df[final_cols].reset_index(drop=True)

In [34]:
# ── 12. SAVE ──────────────────────────────────────────────────────────────────
df_final.to_csv('zus_coffee_enriched.csv', index=False)

In [35]:
# ── 13. SUMMARY ───────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  ZUS COFFEE  DATASET - SUMMARY")
print("="*55)
print(f"Total transactions   : {len(df_final):,}")
print(f"Date range           : {df_final['transaction_date'].min().date()} -> {df_final['transaction_date'].max().date()}")
print(f"\nOutlet zones:")
print(df_final['outlet_zone'].value_counts().to_string())
print(f"\nOrder channels:")
print(df_final['order_channel'].value_counts().to_string())
print(f"\nOccasion flags:")
print(df_final['occasion'].value_counts().to_string())
print(f"\nAvg prep time (all)      : {df_final['prep_time_mins'].mean():.1f} mins")
print(f"Max prep time            : {df_final['prep_time_mins'].max():.1f} mins")
print(f"Avg prep - Normal        : {df_final[df_final['occasion']=='Normal']['prep_time_mins'].mean():.1f} mins")
print(f"Avg prep - Raya 2026     : {df_final[df_final['occasion']=='Raya 2026']['prep_time_mins'].mean():.1f} mins")
print(f"Avg prep - Payday Week   : {df_final[df_final['occasion']=='Payday Week']['prep_time_mins'].mean():.1f} mins")
print(f"\nTop 5 busiest hours:")
print(df_final.groupby('hour')['transaction_id'].count().sort_values(ascending=False).head().to_string())
print("\nFile saved: zus_coffee_enriched.csv")


  ZUS COFFEE  DATASET - SUMMARY
Total transactions   : 146,129
Date range           : 2025-10-01 -> 2026-03-30

Outlet zones:
outlet_zone
Subang / Shah Alam      49708
KLCC / Bukit Bintang    49699
Puchong / Cyberjaya     46722

Order channels:
order_channel
Walk-in       66085
ShopeeFood    47977
GrabFood      32067

Occasion flags:
occasion
Normal            109522
Payday Week        24781
CNY 2026            3265
Raya 2026           3146
New Year 2026       2234
Christmas 2025      2048
Deepavali 2025      1133

Avg prep time (all)      : 12.1 mins
Max prep time            : 51.0 mins
Avg prep - Normal        : 11.8 mins
Avg prep - Raya 2026     : 18.6 mins
Avg prep - Payday Week   : 11.8 mins

Top 5 busiest hours:
hour
10    18017
9     17346
8     17229
7     13086
11     9622

File saved: zus_coffee_enriched.csv


In [36]:
!python 03_sql_analysis.py

python: can't open file 'c:\\Users\\adlee\\OneDrive\\Portfolio\\03_sql_analysis.py': [Errno 2] No such file or directory
